# NB3 · The Knowledge: retrieving from Northbridge's documents

**Agentic Systems Engineering · Lecture 3 · Vizuara · Dr. Rajat Dandekar**

Northbridge Cycles builds e-bikes in Manchester on **Ledgerline ERP 7.2** (1998). Before we change anything, Henrik
Weber, the plant head, has shared 28 years of documentation as a GitHub repository, `northbridge-docs`. Ask
Northbridge must answer staff questions from it, with the document each answer comes from. Two running examples:

> **Emma:** Can I buy the battery packs from Brightcell Energy? They quote £104.
> **Emma:** What does ERR-PO-0412 mean and how do I fix it?

This notebook follows the lecture:

1. clone the dump and audit it: files, classes, tokens
2. parse it two ways and see what the plain parser loses
3. the questions people ask, and the golden set we score against
4. why pasting everything fails
5. chunking three ways
6. BM25 from scratch, scored on the 224 documentation questions
7. a reranker, then embeddings in Chroma, and why exact codes need BM25
8. the shared answer step: Emma's question answered from RAG's passages (recorded, replayed)
9. the LLM Wiki: a real page, search over its sections, and one query's chain (recorded, replayed)
10. the race, read from the log

**No API key needed.** Retrieval runs here, on your machine. Every model call (answers, the judge, the wiki's
navigation) is replayed from our recorded runs, so you see exactly what happened in the race. Four exercises are
marked `TODO (exercise N)`, each with its solution behind `SOLUTION = True`. Set it to `False` and try it yourself.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: replay


In [2]:
# Lecture 3's extra packages: parsers, the tokenizer, Chroma and the embedding and reranking models (about 2 minutes).
import importlib.util
if importlib.util.find_spec("chromadb") is None or importlib.util.find_spec("pdfplumber") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf==5.1.0", "pdfplumber==0.11.4", "openpyxl",
                    "python-docx", "tiktoken", "scipy", "chromadb==1.0.15", "sentence-transformers", "pytesseract"], check=True)
if "google.colab" in sys.modules and subprocess.run(["which", "tesseract"], capture_output=True).returncode:
    subprocess.run("apt-get -qq install -y tesseract-ocr > /dev/null", shell=True)   # OCR for the six scans
os.environ.setdefault("LEDGER_WIKI_PROVIDER", "google-direct")   # where a live wiki call would go (only if you add a key)
SOLUTION = True

In [3]:
import collections
import json
import pathlib
import statistics
import time

from ledgerline.course.l3 import corpus as C
from ledgerline.course.l3 import evaluate as E
from ledgerline.course.l3 import rag


def pp(o, n=1400):
    """Print JSON, cut at n characters."""
    s = json.dumps(o, indent=1, ensure_ascii=False, default=str)
    print(s if len(s) <= n else s[:n] + " …")

## 1 · Clone the dump, audit it

Henrik's repository is public: `github.com/RajatDandekar/northbridge-docs`. `dump_root` clones it next to this
notebook (or uses `$NORTHBRIDGE_DOCS`, if you set it to a copy you already have).

In [4]:
DOCS = C.dump_root(clone_to="northbridge-docs")
print(DOCS)
print((DOCS / "README.md").read_text()[:900])

Cloning into 'northbridge-docs'...


/course/lectures/L3-the-memory/northbridge-docs
# northbridge-docs

**From Henrik Weber, plant head, Northbridge Cycles, Manchester**

Hello, and thank you for taking this on.

You asked for our documentation before you change anything, and you were right to. Here it is: everything we could find about how Northbridge buys, stores, builds, sells and pays, in the shape it is really in. We have run Ledgerline since 1998. The rules live in policies, in procedures, in the vendor's manuals, in spreadsheets people keep on the share, and, if I am honest, in Luis's head and in e-mails that never made it into a policy.

Some of it is out of date. Some of it is there twice. A few documents disagree with each other, and a couple are not in force yet. I have not tidied it, because I think you need to see what we actually work from. Where something is old, the newer document usually says so; where it does not, ask Luis or Daniel.

One folder is pri


In [5]:
files = C.list_files(DOCS)
by_folder = collections.Counter(f.split("/")[0] if "/" in f else "(root)" for f in files)
by_class = collections.Counter(C.FMT_CLASS.get(C.fmt_of(f), "other") for f in files)
print(len(files), "files")
for k, v in sorted(by_folder.items()):
    print(f"  {k:24} {v:4}")
print(dict(by_class))

693 files
  (root)                      1
  01_policies                24
  02_procedures              26
  03_ledgerline_manuals      13
  04_service_definitions    405
  05_master_data             15
  06_transactions            26
  07_data_dictionary          4
  08_release_notes           18
  09_email_archive           79
  10_meeting_notes           32
  11_training                34
  12_integrations            10
  99_scans                    6
{'unstructured': 145, 'structured': 27, 'semi-structured': 521}


Files are not the cost. Tokens are. We parse every file with the **structured** parser (one unit per PDF section,
spreadsheet row, service, e-mail) and count tokens with tiktoken `o200k_base`. The first run parses all 693 files
(a few minutes on Colab); the result is cached.

In [6]:
t0 = time.time()
units = C.parse(DOCS, parser="structured")
print(f"{len(units):,} units in {time.time() - t0:.0f} s; failures: {len(C.parse.failures)}")
tok_by_class = collections.Counter()
for u in units:
    tok_by_class[C.FMT_CLASS.get(u.fmt, "other")] += C.n_tokens(u.text)
total_tokens = sum(tok_by_class.values())
for k, v in tok_by_class.most_common():
    print(f"  {k:16} {v:>10,} tokens  {100 * v / total_tokens:4.1f}%")
print(f"  {'total':16} {total_tokens:>10,}")

51,325 units in 34 s; failures: 0


  structured        3,216,916 tokens  69.3%
  unstructured      1,248,304 tokens  26.9%
  semi-structured     179,389 tokens   3.9%
  total             4,644,609


## 2 · Two parsers, one file

The **plain** parser takes a file's text as it comes. The **structured** parser reads each format properly: e-mail
headers decoded, every spreadsheet row written as `column: value`, scans read by OCR.

In [7]:
for rel in ["09_email_archive/2026/2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml",
            "99_scans/2019_Voltway_code_change_approval_signed.pdf"]:
    for p in ("plain", "structured"):
        us = C.parse_file(DOCS, rel, parser=p)
        text = us[0].text if us else ""
        print(f"--- {p:10} {rel.split('/')[-1]}: {len(us)} unit(s), {C.n_tokens(text)} tokens")
        print(text[:420] or "(no text: an image-only page)")

--- plain      2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml: 1 unit(s), 310 tokens
From: Julien Marchand <j.marchand@brightcell-energy.example>
To: Emma Clarke <emma.clarke@northbridgecycles.example>
Date: Thu, 22 Oct 2026 16:41:00 +0000
Subject: Quotation Q-BE-26-0917 - 36 V 500 Wh battery packs
Message-ID: <c0e8939f459f7898bbeb@mail.northbridgecycles.example>
X-Mailer: NBC Mail Gateway 4.2
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: quoted-printable
MIME-Version: 
--- structured 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml: 1 unit(s), 229 tokens
E-mail. Subject: Quotation Q-BE-26-0917 - 36 V 500 Wh battery packs
From: Julien Marchand <j.marchand@brightcell-energy.example>
To: Emma Clarke <emma.clarke@northbridgecycles.example>
Date: Thu, 22 Oct 2026 16:41:00 +0000

Dear Ms Clarke,

Thank you for your call. As promised, our quotation Q-BE-26-0917 for battery packs, 36 V 500 Wh, compatible with your BAT-500 mounting: £104.00 per pack, 

--- structured 2019_Voltway_code_change_approval_signed.pdf: 1 unit(s), 74 tokens
SUPPLIER MASTER CHANGE REQUEST

Supplier: Voltway Cells, Eindhoven

Old code: VC-EIN-0009 to be BLOCKED
New code: VC-EIN-0042

Reason: Ledgerline 7.1 migration, new supplier numbering
Approved by: G. Sutcliffe

we Vt

Date: 21/03/2019


In [8]:
rows = {p: C.parse_file(DOCS, "05_master_data/SUPPLIER_MASTER.xlsx", parser=p) for p in ("plain", "structured")}
for p, us in rows.items():
    u = next(x for x in us if "VC-EIN-0042" in x.text)
    print(f"{p:10} {u.loc}  {u.text[:240]}")

plain      {'sheet': 'Suppliers', 'row': 4}  VC-EIN-0042 Voltway Cells NL859312774B01 NL Eindhoven ACTIVE N45 BAT MIGRATION 2019-04-01 Femke de Vries orders@vc-ein-0042.example
structured {'sheet': 'Suppliers', 'row': 4}  sup_cd: VC-EIN-0042; name: Voltway Cells; vat_no: NL859312774B01; state: NL; city: Eindhoven; status: ACTIVE; pt_cd: N45; supplies: BAT; created_by: MIGRATION; created_on: 2019-04-01; contact: Femke de Vries; email: orders@vc-ein-0042.examp


## 3 · The questions people ask

The golden set has 304 real-style questions in three kinds. **Genesis** (create something) and **analytical**
(count, sum, compare) need services and tools: Lecture 4 onward. **Documentation** questions are today's.

In [9]:
allq = C.questions(DOCS, category="all")
print(collections.Counter(q["category"] for q in allq))
G = E.Golden(DOCS)
print(len(G.qs), "documentation questions:", dict(collections.Counter(q["type"] for q in G.qs)))
q412 = G.by_id["Q-0412"]
pp({k: q412[k] for k in ("question", "reference_answer", "gold_sources", "must_contain")})

Counter({'documentation': 224, 'genesis': 40, 'analytical': 40})
224 documentation questions: {'multi_hop': 21, 'exact_code': 27, 'single_lookup': 23, 'procedure': 20, 'rule': 25, 'service_field': 32, 'table_lookup': 34, 'version_conflict': 20, 'not_in_docs': 12, 'permission': 10}
{
 "question": "What does ERR-PO-0412 mean and how do I fix it?",
 "reference_answer": "ERR-PO-0412 TAX CODE DOES NOT MATCH SUPPLIER COUNTRY: the tax code on the order is for the wrong country (the supplier's country code is shown in brackets). A UK supplier needs a -UK code; a supplier in any other country needs a reverse-charge -RC code. Fetch always fills VAT20-UK, so for an overseas supplier such as Voltway Cells (NL) change the tax code to the -RC code with the same rate, for example VAT20-RC, and save again.",
 "gold_sources": [
  {
   "file": "03_ledgerline_manuals/MAN-PUR-72 Ledgerline 7.2 User Manual - Purchasing v7.2.3.pdf",
   "fact_id": "F-MAN-17.2",
   "page": 10,
   "section": "17.2 ERR-PO-0412 

## 4 · Why not paste everything?

Gemini 3.8 Flash's window is 1,048,576 tokens (OpenRouter's listing on 3 Nov 2026; the largest window listed was
2,000,000). Our dump is:

In [10]:
print(f"{total_tokens:,} tokens = {total_tokens / 1_048_576:.1f} x Gemini 3.8 Flash's window, {total_tokens / 2_000_000:.1f} x the largest")
print(f"input cost of one question, if it fitted: ${total_tokens * 0.825 / 1e6:.2f} (at $0.825 per million input tokens)")

4,644,609 tokens = 4.4 x Gemini 3.8 Flash's window, 2.3 x the largest
input cost of one question, if it fitted: $3.83 (at $0.825 per million input tokens)


## 5 · Chunking, three ways

A chunk is what gets indexed and searched. Fixed windows cut wherever 200 tokens end; recursive splitting prefers
paragraph and line breaks; structure-aware chunking packs whole sections and never cuts one unless it must.

In [11]:
policy = [u for u in units if u.file == "01_policies/POL-PUR-001_Purchase_Policy_v4.1.pdf"]
for m in ("fixed", "recursive", "structure"):
    ch = C.chunk(policy, m, 200, 0.0)
    print(f"--- {m}: {len(ch)} chunks")
    for c in ch[2:4]:
        print(f"  [{c.cite}] {c.n_tokens} tokens: {c.body[:110]!r}")

--- fixed: 11 chunks
  [POL-PUR-001 v4.1, pp.1-2, § 3. Requests before orders] 235 tokens: '3 A direct purchase order without a request is allowed only when its gross value, including VAT, is\n£50,000 or'
  [POL-PUR-001 v4.1, p.2, § 5. Approval limits] 234 tokens: 'approved by the next person up the line whose limit covers it.\n5.3 Nobody may approve an order they raised.\n5.'
--- recursive: 12 chunks
  [POL-PUR-001 v4.1, p.1, § 3. Requests before orders] 213 tokens: '3.1 Every purchase starts with a purchase request (PR) in Ledgerline, except a direct order under §3.3.\n3.2 A '
  [POL-PUR-001 v4.1, pp.1-2, § 4. Cost centres and budgets] 217 tokens: "4.3 Before raising an order, check the cost centre's status in Ledgerline (screen FIN-CC-01) or ask\nFinance.\n5"
--- structure: 15 chunks
  [POL-PUR-001 v4.1, p.1, § 1. Purpose] 225 tokens: '1. Purpose\nThis policy sets out how Northbridge Cycles buys materials, components and services, so that every\n'
  [POL-PUR-001 v4.1, p.1, § 4. Cost

In [12]:
t0 = time.time()
chunks = C.chunk(units, "structure", 200, 0.15)   # the race winner's chunking: structured-structure-200-o15
print(f"{len(chunks):,} chunks in {time.time() - t0:.0f} s, {statistics.mean(c.n_tokens for c in chunks):.0f} tokens on average")
by_id = {c.cid: c for c in chunks}

28,953 chunks in 10 s, 196 tokens on average


## 6 · BM25 from scratch

BM25 scores a chunk by the query words it contains, weighted by how rare each word is (idf) and damped for long
chunks. Our tokenizer keeps codes like `ERR-PO-0412` whole. Here is the scoring code we use, then a search.

In [13]:
import inspect
print(inspect.getsource(rag.tokenize))
print(rag.tokenize("What does ERR-PO-0412 mean? Rule 6.2, UN 38.3, BAT-500"))

def tokenize(text: str) -> list[str]:
    """Lower-case words; codes like ERR-PO-0412, BAT-500, UN38.3 and section numbers like 6.2 survive whole,
    and their parts are added too (so 'po' and '0412' also match)."""
    out = []
    for t in re.findall(r"[a-z0-9£]+(?:[.\-/_][a-z0-9]+)*", (text or "").lower()):
        if t in STOP:
            continue
        out.append(_stem(t))
        if any(c in t for c in "-/_."):
            out.extend(_stem(p) for p in re.split(r"[-/_.]", t) if len(p) > 1 and p not in STOP)
    return out

['err-po-0412', 'err', 'po', '0412', 'mean', 'rule', '6.2', 'un', '38.3', '38', 'bat-500', 'bat', '500']


In [14]:
idx = rag.Index("/tmp/nb3_index", chunks)
t0 = time.time()
bm = idx.bm25
print(f"BM25 over {len(chunks):,} chunks built in {time.time() - t0:.1f} s, {len(bm.vocab):,} terms")
t0 = time.perf_counter()
hits = bm.search("What does ERR-PO-0412 mean and how do I fix it?", 5, ~idx.conf)
print(f"search: {1000 * (time.perf_counter() - t0):.1f} ms")
for i, s in hits:
    print(f"  {s:6.2f}  {chunks[i].cite}")

BM25 over 28,953 chunks built in 3.9 s, 91,258 terms
search: 1.4 ms
   39.09  QRC-03 v2026.1, p.1, § Quick reference: ERR-PO-0412 quick fix
   35.73  QRC-03 v2026.1, p.1, § Reading the country indicator
   35.64  QRC-03 v2026.1, p.1, § Recommitting the record
   31.86  New starter - Purchasing (2024) deck export.pdf, pp.6-8, § Tax codes
   29.37  SOP-PUR-002 v1.4, p.3, § 5. Procedure


Now score BM25 alone on all 224 documentation questions: is a gold source in the top 5? (`not_in_docs` and refused
permission questions are left out of retrieval scoring, so 205 count.)

In [15]:
bm25_cfg = rag.Config(parser="structured", chunker="structure", size=200, overlap=0.15, retrieval="bm25", rerank=False, k=10)
bm25_only = rag.Retriever(idx, bm25_cfg)
rep = E.retrieval_only(lambda q, role: bm25_only.search(q, role=role), G)
s = rep["summary"]["overall"]
print(f"BM25 alone: recall@5 {s['recall@5']:.3f}, recall@10 {s['recall@10']:.3f}, MRR {s['mrr']:.3f}; "
      f"search p95 {1000 * rep['summary']['latency_s']['p95']:.1f} ms")

BM25 alone: recall@5 0.771, recall@10 0.873, MRR 0.597; search p95 0.5 ms


The grid measured this exact configuration (structured, structure-aware 200 tokens, 15% overlap, BM25 alone) at
recall@5 0.771 and recall@10 0.873: the same numbers, from a fresh clone.

### TODO (exercise 1) · Which question types does BM25 miss?
Print recall@5 for each question type from `rep["summary"]["by_type"]`, worst first.

In [16]:
if SOLUTION:
    for t, v in sorted(rep["summary"]["by_type"].items(), key=lambda x: x[1].get("recall@5") or 9):
        if v.get("recall@5") is not None:
            print(f"  {t:18} n={v['n']:3}  recall@5 {v['recall@5']:.2f}")

  exact_code         n= 27  recall@5 0.63
  table_lookup       n= 34  recall@5 0.65
  version_conflict   n= 20  recall@5 0.70
  multi_hop          n= 21  recall@5 0.76
  procedure          n= 20  recall@5 0.80
  single_lookup      n= 23  recall@5 0.83
  rule               n= 25  recall@5 0.88
  service_field      n= 32  recall@5 0.91
  permission         n=  3  recall@5 1.00


## 7 · The reranker, then embeddings in Chroma

A **cross-encoder** reranker (`BAAI/bge-reranker-base`) reads the question and each candidate together and scores
the pair. It is slower than BM25, so it only reorders BM25's top 20. This is the race winner's configuration.

In [17]:
best_cfg = rag.best_config()
print(best_cfg.id)
best = rag.Retriever(idx, best_cfg)
for qid in ("Q-0412", "Q-EMMA"):
    q = G.by_id[qid]
    out = best.search(q["question"], role=C.role_of(q))
    print(f"--- {qid}: {q['question']}  ({1000 * out['seconds']:.0f} ms on this machine; timing {out['timing']})")
    for it in out["items"][:5]:
        print(f"  {it['rank']:2}. {it['cite']}")

structured-structure-200-o15|-|bm25|rr1|qr0|k10


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Q-0412: What does ERR-PO-0412 mean and how do I fix it?  (5199 ms on this machine; timing {'bm25': 0.00077, 'rerank': 5.19784})
   1. New starter - Purchasing (2024) deck export.pdf, pp.6-8, § Tax codes
   2. QRC-03 v2026.1, p.1, § Recommitting the record
   3. QRC-03 v2026.1, p.1, § Reading the country indicator
   4. QRC-03 v2026.1, p.1, § Quick reference: ERR-PO-0412 quick fix
   5. RN-7.2.3-P1, p.1, § Overview of Changes


--- Q-EMMA: Can I buy the battery packs from Brightcell Energy? They quote £104.  (4622 ms on this machine; timing {'bm25': 0.00114, 'rerank': 4.62066})
   1. 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml, § body
   2. 2026-10-23_RE_ Brightcell quote - can we switch from Voltway_.eml, § body
   3. 2026-10-23_Brightcell quote - can we switch from Voltway_.eml, § body
   4. 2026-10-19 Weekly purchasing meeting.md, § Weekly purchasing meeting
   5. 2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml, § body


On a CPU the cross-encoder takes seconds per question; the race ran it on an L4 GPU, where the whole search took
48 ms at p95. On 40 questions, how much does it add over BM25 alone? (On a CPU this takes a few minutes.)

In [18]:
sub = G.qs[::5][:40]
a = E.retrieval_only(lambda q, role: bm25_only.search(q, role=role), G, sub)["summary"]["overall"]
b = E.retrieval_only(lambda q, role: best.search(q, role=role), G, sub)["summary"]["overall"]
print(f"40 questions: recall@5 BM25 {a['recall@5']:.3f} -> BM25 + reranker {b['recall@5']:.3f}")

40 questions: recall@5 BM25 0.722 -> BM25 + reranker 0.861


**Embeddings.** A dense retriever turns every chunk into a vector (here `bge-small`, 384 numbers) and finds the
nearest vectors to the question's. We embed a small part of the dump (the manuals, the training cards and the
policies) into a persistent **Chroma** collection, with HNSW set to search widely (`ef_search` 4000, as we measured).

In [19]:
import chromadb
part = [c for c in chunks if c.file.split("/")[0] in ("01_policies", "03_ledgerline_manuals", "11_training") and not c.meta.get("confidential")]
emb = rag.Embedder("bge-small")
t0 = time.time()
vecs = emb.passages([c.text for c in part])
print(f"{len(part):,} chunks embedded in {time.time() - t0:.0f} s, vectors {vecs.shape}")
client = chromadb.PersistentClient(path="/tmp/nb3_chroma")
col = client.get_or_create_collection("bge-small", configuration={"hnsw": dict(rag.HNSW)})   # cosine, ef_search 4000
if col.count() == 0:
    for i in range(0, len(part), 4000):
        col.add(ids=[str(j) for j in range(i, min(i + 4000, len(part)))], embeddings=vecs[i:i + 4000].tolist())
print(col.count(), "vectors in Chroma")

2,364 chunks embedded in 152 s, vectors (2364, 384)


2364 vectors in Chroma


In [20]:
def dense(question, k=5):
    r = col.query(query_embeddings=[emb.query(question).tolist()], n_results=k, include=["distances"])
    return [(part[int(i)].cite, round(1 - d, 3)) for i, d in zip(r["ids"][0], r["distances"][0])]

for question in ["What does ERR-PO-0412 mean and how do I fix it?", "Can we buy battery packs from a supplier without a UN 38.3 certificate?"]:
    print("---", question)
    for cite, sim in dense(question):
        print(f"  {sim:.3f}  {cite}")

--- What does ERR-PO-0412 mean and how do I fix it?
  0.712  MAN-ERR-72 v7.2.3, p.13, § ERR-MAT-0412 NO GRN FOR PO LINE
  0.706  error_messages.html, § Error messages
  0.697  MAN-ERR-72 v7.2.3, pp.16-17, § ERR-PER-0010 UNPOSTED DOCUMENTS IN PERIOD
  0.697  MAN-SVC-72 v7.2.3, p.8, § po_cancel PO CANCEL
  0.696  MAN-ERR-72 v7.2.3, p.25, § ERR-SYS-0422 UNKNOWN FIELD <FIELD> / BAD VALUE FOR <FIELD>
--- Can we buy battery packs from a supplier without a UN 38.3 certificate?
  0.853  POL-SUP-003 v2.2, § 4. UN 38.3 certificates
  0.852  POL-SUP-003 v2.3, p.2, § 4. UN 38.3 certificates
  0.852  POL-SUP-003 v2.3, p.2, § 4. UN 38.3 certificates
  0.785  POL-PUR-001 v4.1, p.2, § 6. Supplier qualification
  0.771  POL-PUR-001 v4.1, p.3, § What are the rules before ordering battery packs, hub motors


### TODO (exercise 2) · BM25 against dense on the same part of the dump
Build a BM25 over `part` (`rag.BM25([c.text for c in part])`) and print its top 5 for the ERR-PO-0412 question
next to the dense top 5. Which one finds QRC-03 first?

In [21]:
if SOLUTION:
    bm_part = rag.BM25([c.text for c in part])
    for i, s in bm_part.search("What does ERR-PO-0412 mean and how do I fix it?", 5):
        print(f"  {s:6.2f}  {part[i].cite}")

   30.21  QRC-03 v2026.1, p.1, § Quick reference: ERR-PO-0412 quick fix
   27.66  QRC-03 v2026.1, p.1, § Reading the country indicator
   27.59  QRC-03 v2026.1, p.1, § Recommitting the record
   24.08  New starter - Purchasing (2024) deck export.pdf, pp.6-8, § Tax codes
   19.29  error_messages.html, § Error messages


## 8 · The shared answer step (recorded, replayed)

Every method in the race hands its passages to the same answer prompt: answer only from the numbered sources, cite
each fact, give the rule in force today, refuse confidential HR content. We rebuild the race's prompt for Emma's
question from the exact passages RAG retrieved in the race (`race_examples.json` lists their chunk ids), and replay
the recorded answer and judge calls.

In [22]:
RX = json.loads((pathlib.Path(rag.__file__).parent / "race_examples.json").read_text())
from ledgerline.course.l3.wiki_llm import WikiLLM
ans_llm = WikiLLM("L3v4_race_rag_answers", max_tokens=2048)
judge_llm = WikiLLM("L3v4_race_rag_judge", max_tokens=512)
q = G.by_id["Q-EMMA"]
items = [by_id[i].as_item() for i in RX["questions"]["Q-EMMA"]["items"]]
msgs = E.answer_messages(q, items, answer_words=RX["answer_words"])
print(msgs[0]["content"])
print("…", len(msgs[1]["content"]), "characters of sources")

You are Ask Northbridge, the documentation assistant of Northbridge Cycles (Manchester), who run Ledgerline ERP 7.2.
Today is Monday 26 October 2026. You are answering Emma Clarke (ECLARKE), purchase executive.
Answer ONLY from the numbered sources in the user message. Cite every fact with its source number in square brackets, like [2] or [1][4].
- Give the rule in force today. A source marked SUPERSEDED, an old copy, or NOT YET IN FORCE does not apply today; if it matters, say what changes and when.
- If an amendment, release note or e-mail changed what an older document says, give the current position and mention the change.
- If the sources do not answer the question, reply "NOT IN THE DOCUMENTS." and one sentence on what is missing. Never guess.
- If the question asks for confidential HR information that this user may not see, refuse politely and say who may see it.
Be concise: at most 250 words, plain sentences, exact codes and numbers as the sources give them.
… 7786 characters o

In [23]:
a = E.answer(q, items, ans_llm, answer_words=RX["answer_words"])
print(a["answer"])
print(f"\nreplayed: {a['replayed']} · {a['tokens_in']} tokens in · {a['llm_seconds']} s when it was recorded")
print("exact-fact check:", E.fact_check(q, a["answer"], G))
print("judge:", E.judge(q, a["answer"], judge_llm, G))

No, you cannot order the battery packs from Brightcell Energy today [2][6]. 

The reasons are:
1. **Not an active supplier:** Under Purchase Policy 6.1, orders may only be placed with active suppliers [10]. Brightcell is not in the supplier master and must first complete onboarding under SOP-PUR-001, requiring Daniel Okafor’s signature on form NBC-F-112 [4][6].
2. **No valid UN 38.3 report:** Battery packs are lithium-cell parts governed by Purchase Policy 6.2 [2][10]. As amended by AMD-SUP-2026-02, a report must be on file, less than 12 months old, and valid for at least 60 days after the order [2]. Brightcell’s report is currently being renewed and is not expected until December 2026 [1][2].
3. **Cheaper quote restriction:** Under Policy 6.4, a cheaper quotation alone is not permitted as a reason to switch suppliers for a lithium-cell part [2][10].

Note that although Ledgerline ERP will not block the order because the system certificate check is switched off, a clean save does not a

Which of the question's 11 gold sources did RAG bring back? The cites tell you why the answer misses £104 against
£119 and PUR.CERT_CHK.

In [24]:
for it in items:
    print(" ", it["cite"])
print("gold:", [g["file"].split("/")[-1] for g in G.gold(q)])

  2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml, § body
  2026-10-23_RE_ Brightcell quote - can we switch from Voltway_.eml, § body
  2026-10-23_Brightcell quote - can we switch from Voltway_.eml, § body
  2026-10-19 Weekly purchasing meeting.md, § Weekly purchasing meeting
  2026-10-22_Quotation Q-BE-26-0917 - Brightcell Energy.eml, § body
  2026-10-23_RE_ Brightcell quote - can we switch from Voltway_.eml, § body
  SOP-SAL-001 v2.2, p.4, § What to do when something does not look right
  POL-BAT-004 v1.2, pp.1-2, § Common errors and misunderstandings
  SOP-INV-004 v1.0, p.3, § Common Errors
  POL-PUR-001 v4.1, p.3, § What are the rules before ordering battery packs, hub motors
gold: ['POL-PUR-001_Purchase_Policy_v4.1.pdf', 'SOP-PUR-001 Supplier Onboarding v3.1.pdf', 'SOP-PUR-001 Supplier Onboarding v3.1.pdf', 'SOP-PUR-001 Supplier Onboarding v3.1.pdf', 'AMD-SUP-2026-02 Amendment 2 Supplier Policy.pdf', 'POL-PUR-001_Purchase_Policy_v4.1.pdf', 'RN-7.2.3-P4 Ledgerline Patch N

### TODO (exercise 3) · The same step for ERR-PO-0412
Rebuild the items for `Q-0412` from `RX`, replay the answer with `ans_llm`, and run the exact-fact check.

In [25]:
if SOLUTION:
    q2 = G.by_id["Q-0412"]
    items2 = [by_id[i].as_item() for i in RX["questions"]["Q-0412"]["items"]]
    a2 = E.answer(q2, items2, ans_llm, answer_words=RX["answer_words"])
    print(a2["answer"][:900])
    print(E.fact_check(q2, a2["answer"], G))

**Meaning**  
`ERR-PO-0412` displays as `TAX CODE DOES NOT MATCH SUPPLIER COUNTRY (XX)` [4][5]. It means the entered tax code does not match the country recorded on the supplier account [4][5][6]. The letters in brackets show the supplier's country code (e.g., NL for the Netherlands) [3][4][5]. This occurs because Fetch always automatically populates `VAT20-UK`, even for overseas suppliers [4][6][8].

**How to Fix It**  
1. **Confirm the supplier location:** Check the country code shown in brackets at the end of the error message [3][4].
2. **Amend the tax code:** Check each purchase order line item individually [3]:
   - **Outside the UK:** Change the tax code to `VAT20-RC` for standard goods/materials, or `VAT5-RC` for reduced-rate items [3][4].
   - **UK supplier:** Use `VAT20-UK` [3][4].
3. **Resave:** With the cursor remaining in the line details area, press **F8** again to run the 
{'ok': True, 'expected': 'answer', 'missing': []}


## 9 · The LLM Wiki

Karpathy's LLM Wiki ("LLM Wiki", a gist, April 2026): an LLM reads the raw sources once and writes a wiki of linked
pages, which it then maintains. Three layers: the raw sources (never edited), the wiki (pages the LLM owns, an
`index.md` and a `log.md`) and the schema (`AGENTS.md`). Three operations: ingest, query, lint. Our wiki ships with
the course package.

In [26]:
from ledgerline.course.l3 import wiki_query as WQ
w = WQ.Wiki(dump=DOCS)
print(len(w.pages), "pages")
print((w.root / "AGENTS.md").read_text()[:700])
print(w.page_text("sup-brightcell-energy")[:1600])

683 pages
# AGENTS.md: the schema of Northbridge's wiki

This file is the wiki's constitution. Every agent that ingests, queries or lints the wiki reads it first.
It follows Karpathy's "LLM Wiki" pattern (gist, 4 Apr 2026): **raw sources** (immutable), **the wiki**
(markdown pages the LLM owns) and **this schema**. Obsidian is the IDE, the LLM is the programmer, the wiki
is the codebase.

Today, for every rule below, is **Monday 26 October 2026, 09:00 GMT** (the date the dump was taken).

## 1. The three layers

| Layer | Where | Who writes it |
|---|---|---|
| Raw sources | `northbridge-docs/` (699 files: PDF, DOCX, XLSX, CSV, YAML, XML, EML, MD, HTML, scans) | nobody: never edited |
| The wiki | `wi
# Brightcell Energy [[sup-brightcell-energy]]
| | |
|---|---|
| Supplier | Brightcell Energy |
| Location | Lyon, France |
| Contact | Julien Marchand, Key Account Manager |
| Status | Prospective / Not onboarded |
| Key items | 36 V 500 Wh battery packs |
| Quotation | Q-BE-26-0917 |

Br

In [27]:
print((w.root / "index.md").read_text()[:700])
print([l for l in (w.root / "log.md").read_text().splitlines() if l.startswith("## [")][:3])

# Northbridge wiki: index

679 pages. Read this first; open a page with its slug. Format: `[[slug]] Title: summary`. Pages named by a code follow the slug rules below; open the list page to see them all.

## Policies and amendments (13)

- [[amd-pur-2026-03]] AMD-PUR-2026-03 Amendment 3 to the Purchase Policy: Amendment reducing the direct purchase order threshold from £50,000 to £25,000 gross effective 1 December …
- [[amd-sup-2026-02]] AMD-SUP-2026-02 Amendment 2 to the Supplier Policy: Amendment to POL-SUP-003 tightening UN 38.3 battery certificate validity to within 12 months of issue and 60 …
- [[pol-abc-011]] POL-ABC-011 Anti-Bribery and Gifts Policy: Anti-bribery and gifts policy sett
['## [2026-11-03T05:31:37Z] ingest | README.md | +2 ~0 | data-nbc-change-log, user-hweber', '## [2026-11-03T05:30:21Z] ingest | 05_master_data/BOM_master_2026.xlsx | +10 ~0 | data-bom-master, items-fg, items-bat, items-mtr, items-chg, items-mnt, items-bel, items-brk, items-pkg, items-box', '## [202

**Search over the wiki's sections** (no LLM): BM25 and bge-small over every section of every page, fused.

In [28]:
si = WQ.SectionIndex(w)
for hit in si.search("Can I buy the battery packs from Brightcell Energy? They quote £104.", role="PUR-EXEC", k=5):
    print(f"  [[{hit['page']}]] {hit['title']} > {hit['section']}")

  [[sup-brightcell-energy]] Brightcell Energy > Rate contracts
  [[sup-brightcell-energy]] Brightcell Energy > Record
  [[topic-un-38-3-certificates]] UN 38.3 Certificates > Purchasing and Quotation Controls
  [[items-bat]] BAT-* items > Procurement and contracts
  [[user-eclarke]] Emma Clarke (ECLARKE) > Purchasing operations


**Query by navigation** (recorded, replayed): the model reads `index.md`, opens pages, follows links, and stops when
it has enough. Each step is a model call.

In [29]:
nav_llm = WikiLLM("L3v4_wiki_nav", max_tokens=4000)
out = WQ.Navigator(w, nav_llm).navigate(q["question"], "PUR-EXEC")
for s in out["chain"]:
    print(" ", s)
print(len(out["items"]), "pages go to the shared answer step")

  {'step': 'index', 'what': 'index.md'}
  {'step': 'page', 'what': 'sup-brightcell-energy', 'turn': 1, 'from': None}
  {'step': 'page', 'what': 'pol-sup-003', 'turn': 1, 'from': None}
  {'step': 'page', 'what': 'sop-pur-006', 'turn': 1, 'from': None}
  {'step': 'done', 'why': 'Brightcell Energy is not onboarded in the supplier master, has no active rate contract, and is legally/operationally blocked from onboarding or receiving POs under POL-SUP-003, AMD-SUP-2026-02, and SO', 'turn': 2}
3 pages go to the shared answer step


## 10 · The race, from the log

The full race ran on Modal: 224 questions, four methods, the same answer prompt (250 words), the same model and
judge. Its log is in the course repository at `lectures/L3-the-memory/experiments/logs/race.json`.

In [30]:
import urllib.request
RACE_URL = "https://raw.githubusercontent.com/RajatDandekar/agents-for-enterprises/main/lectures/L3-the-memory/experiments/logs/race.json"
local = pathlib.Path("experiments/logs/race.json")   # when you run it inside the course repository
race = json.loads(local.read_text()) if local.exists() else json.loads(urllib.request.urlopen(RACE_URL).read())
print(f"{'method':10} {'judge':>6} {'facts':>6} {'cites':>6} {'search p95':>11} {'answer p50':>11} {'$/q':>8}")
for m, o in race["overall"].items():
    print(f"{m:10} {o['answer_correct']:6.1%} {o['fact_correct']:6.1%} {o['citation_correct']:6.1%} "
          f"{o['retrieval_p95_s'] * 1000:9.0f}ms {o['answer_p50_s']:10.1f}s {o['cost_per_q_usd']:8.4f}")
print("winner:", race["decision"]["winner"])

method      judge  facts  cites  search p95  answer p50      $/q
rag         89.3%  85.7%  92.2%        48ms        6.2s   0.0046
wiki        89.3%  83.5%  85.9%     23441ms       12.5s   0.0322
hybrid      92.9%  86.6%  91.7%     11217ms       12.6s   0.0242
sections    79.5%  76.3%  79.5%         1ms        6.5s   0.0054
winner: rag


### TODO (exercise 4) · Your own question
Write a question Emma might ask, search it with `best` (RAG) and with `si` (the wiki's sections), and compare the
top 3 of each. Which one would you trust, and why?

In [31]:
MY_Q = "Who must approve a purchase order of £300,000?"
if SOLUTION:
    for it in best.search(MY_Q, role="PUR-EXEC")["items"][:3]:
        print("  RAG ", it["cite"])
    for hit in si.search(MY_Q, role="PUR-EXEC", k=3):
        print("  wiki", f"[[{hit['page']}]] {hit['section']}")

  RAG  POL-PUR-001 v3.0, § 3. Requests before orders
  RAG  POL-APR-002 v3.0, p.1, § 3. The limits
  RAG  POL-APR-002 v3.0, p.2, § 5. Examples
  wiki [[role-pur-mgr]] Responsibilities and authority
  wiki [[screen-pur-poa-04]] Messages
  wiki [[sop-pur-002]] Steps
